In [0]:
%pip install python-dotenv
%pip install groq
%pip install tabulate
%restart_python

In [0]:
import json
import os
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

try:
    GROQ_API_KEY = os.getenv("GROQ_API_KEY") 
    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY not found")

    daily_perf_df = spark.sql("""
        SELECT *
        FROM payment_gateway_catalog.gold.gold_gateway_daily_performance
        ORDER BY transaction_date DESC, provider
    """)

    failure_df = spark.sql("""
        SELECT *
        FROM payment_gateway_catalog.gold.gold_failure_root_causes
        ORDER BY occurence_count DESC
    """)

    perf_markdown = daily_perf_df.toPandas().to_markdown(index=False)
    failures_markdown = failure_df.toPandas().to_markdown(index=False)

    system_prompt = """You are a Principal Financial Risk and Reliability Engineer. Analyze the provided daily payment gateway summary and failure metrics.
    Generate an executive briefing with three distinct sections:
    1. Operational Health & Gateway Drop-offs: Benchmark success rates (Flag any provider < 90%).
    2. Revenue at Risk & Failure Clustering: Identify the primary failure mode causing the largest financial drag.
    3. Corrective Action Plan: List concrete technical/operational tasks for the integration team.
    """

    user_prompt = f"""
    ### 1. Gold Gateway Daily Performance:
    {perf_markdown}

    ### 2. Top Gateway Failures:
    {failures_markdown}
    """
    model="openai/gpt-oss-20b"
    groq = Groq(api_key=GROQ_API_KEY)
    response = groq.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ], temperature=0.2
    )
    print(response.choices[0].message.content)

except Exception as e:
    print(f"Error executing AI Agent Pipeline: {e}")